# Translate Knowledge Tuning Flows to a New Language

## Overview

This notebook translates the English [Enhanced Multi-Summary QA](../../../src/sdg_hub/flows/qa_generation/document_grounded_qa/enhanced_multi_summary_qa/) knowledge tuning flows to a target language using the sdg_hub **Prompt Translation Flow**.

It produces a complete set of translated flows (11 files) that are immediately discoverable by `FlowRegistry`. After running this notebook, you can set `SDG_LANG=<YourLanguage>` in your `.env` and run [knowledge_generation.ipynb](knowledge_generation.ipynb) to generate SDG data in the target language.

## What Gets Translated

- **7 prompt YAML files** — LLM instructions, examples, and guidelines are translated
- **4 flow YAML files** — Metadata (name, description, tags) and prompt references are updated

## What Does NOT Get Translated

- Jinja2 template variables (`{{document}}`, `{{question}}`, etc.)
- Parsing tags (`[QUESTION]`, `[END]`, `[Start of Context]`, etc.)
- Block types, block names, column names, and all pipeline configuration

## Output Location

Translated flows are written to a **local directory** (`./translated_flows/<language>/`) in your working directory, not into the installed package. This works whether sdg_hub was installed via pip or from a local clone. The notebook registers this directory with `FlowRegistry` so the flows are immediately discoverable.

## Important

After translation, **review the output prompt files** for quality before using them for data generation. LLM-based translation is a starting point — domain-specific terms or nuanced instructions may need manual refinement.

In [ ]:
import os
import copy
from pathlib import Path

import pandas as pd
import yaml
from dotenv import load_dotenv

from sdg_hub import Flow, FlowRegistry

load_dotenv()

# Required for async flow execution in notebooks
import nest_asyncio
nest_asyncio.apply()

# ============================================================================
# CONFIGURATION — Set SDG_LANG and SDG_LANG_CODE in your .env file
# ============================================================================

TARGET_LANGUAGE = os.getenv("SDG_LANG", "").strip()
LANGUAGE_CODE = os.getenv("SDG_LANG_CODE", "").strip()

assert TARGET_LANGUAGE, "SDG_LANG must be set in .env (e.g., 'French', 'Spanish')"
assert LANGUAGE_CODE, "SDG_LANG_CODE must be set in .env (e.g., 'fr', 'es')"

# Output directory for translated flows (local to your working directory)
# Set TRANSLATED_FLOWS_DIR in .env to customize, or leave default
OUTPUT_BASE = Path(os.getenv("TRANSLATED_FLOWS_DIR", "./translated_flows")).resolve()
OUTPUT_DIR = OUTPUT_BASE / TARGET_LANGUAGE.lower()

max_concurrency = int(os.getenv("MAX_CONCURRENCY", "50"))

print(f"Target language: {TARGET_LANGUAGE} ({LANGUAGE_CODE})")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# Setup model configuration for the translation flow
# Uses dedicated TRANSLATION_MODEL if set, otherwise falls back to the main MODEL_PROVIDER config.
# This allows using a different model for translation than for data generation (the teacher model).
def set_translation_model_config(flow_object):
    translation_model = os.getenv("TRANSLATION_MODEL", "").strip()

    if translation_model:
        # Use dedicated translation model
        kwargs = {"model": translation_model}
        api_key = os.getenv("TRANSLATION_API_KEY", "").strip()
        api_base = os.getenv("TRANSLATION_API_BASE", "").strip()
        if api_key:
            kwargs["api_key"] = api_key
        if api_base:
            kwargs["api_base"] = api_base
        flow_object.set_model_config(**kwargs)
        print(f"Using translation model: {translation_model}")
    else:
        # Fall back to main MODEL_PROVIDER config
        model_provider = os.getenv("MODEL_PROVIDER", "hosted_vllm")
        print(f"TRANSLATION_MODEL not set, falling back to MODEL_PROVIDER: {model_provider}")
        if model_provider == "hosted_vllm":
            flow_object.set_model_config(
                model=os.getenv("VLLM_MODEL", "hosted_vllm/meta-llama/Llama-3.3-70B-Instruct"),
                api_base=os.getenv("VLLM_API_BASE", "http://localhost:8000/v1"),
                api_key=os.getenv("VLLM_API_KEY", "EMPTY"),
            )
        elif model_provider == "openai":
            flow_object.set_model_config(
                model=os.getenv("OPENAI_MODEL", "openai/gpt-4"),
                api_key=os.getenv("OPENAI_API_KEY"),
            )
        elif model_provider == "openrouter":
            flow_object.set_model_config(
                model=os.getenv("OPENAI_MODEL", "openai/gpt-4"),
                api_key=os.getenv("OPENAI_API_KEY"),
                api_base="https://openrouter.ai/api/v1",
            )
        elif model_provider == "ollama":
            flow_object.set_model_config(
                model=os.getenv("OLLAMA_MODEL", "ollama/gemma2"),
                api_base=os.getenv("OLLAMA_API_BASE", "http://localhost:11434"),
            )
        elif model_provider == "maas":
            flow_object.set_model_config(
                model=os.getenv("MAAS_MODEL"),
                api_base=os.getenv("MAAS_API_BASE"),
                api_key=os.getenv("MAAS_API_KEY"),
            )
    return flow_object

In [ ]:
# Load the Prompt Translation Flow from the registry
FlowRegistry.discover_flows()

translation_flow_path = FlowRegistry.get_flow_path("Prompt Translation Flow")
assert translation_flow_path, "Could not find Prompt Translation Flow in registry"

translation_flow = Flow.from_yaml(translation_flow_path)
translation_flow = set_translation_model_config(translation_flow)
translation_flow.print_info()

In [ ]:
# Locate the English source flows
english_flow_path = FlowRegistry.get_flow_path(
    "Extractive Summary Knowledge Tuning Dataset Generation Flow"
)
assert english_flow_path, "Could not find English extractive summary flow in registry"

SOURCE_DIR = Path(english_flow_path).parent.parent  # enhanced_multi_summary_qa/

print(f"Source directory: {SOURCE_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

if OUTPUT_DIR.exists():
    print(f"\nOutput directory already exists. Files will be overwritten.")

In [ ]:
# Define source file mappings
# Shared prompt configs (in parent directory, referenced via ../)
SHARED_PROMPTS = {
    "evaluate_faithfulness.yaml": f"evaluate_faithfulness_{LANGUAGE_CODE}.yaml",
    "generate_answers.yaml": f"generate_answers_{LANGUAGE_CODE}.yaml",
    "generate_multiple_qa.yaml": f"generate_multiple_qa_{LANGUAGE_CODE}.yaml",
    "generate_question_list.yaml": f"generate_question_list_{LANGUAGE_CODE}.yaml",
}

# Flow-specific prompt configs (in sub-directories)
FLOW_SPECIFIC_PROMPTS = {
    "extractive_summary/extractive_summary.yaml": f"extractive_summary/extractive_summary_{LANGUAGE_CODE}.yaml",
    "detailed_summary/detailed_summary.yaml": f"detailed_summary/detailed_summary_{LANGUAGE_CODE}.yaml",
    "key_facts/key_facts_summary.yaml": f"key_facts/key_facts_summary_{LANGUAGE_CODE}.yaml",
}

# Flow definition files (not translated, just adapted)
FLOW_FILES = [
    "extractive_summary/flow.yaml",
    "detailed_summary/flow.yaml",
    "key_facts/flow.yaml",
    "doc_direct_qa/flow.yaml",
]

all_prompts = {**SHARED_PROMPTS, **FLOW_SPECIFIC_PROMPTS}
print(f"Prompt files to translate: {len(all_prompts)}")
print(f"Flow files to adapt: {len(FLOW_FILES)}")
print(f"Total files to create: {len(all_prompts) + len(FLOW_FILES)}")

## Step 1: Translate Prompt YAML Files

Each prompt YAML is a list of messages (`role` + `content`). We:
1. Extract all message contents into a flat dataset
2. Run the **Prompt Translation Flow** on the entire dataset (leveraging async batch processing)
3. Reconstruct the translated YAML files

In [ ]:
# Pre-process: extract all prompt message contents into a flat dataset
rows = []
for source_rel, output_rel in all_prompts.items():
    source_path = SOURCE_DIR / source_rel
    with open(source_path) as f:
        messages = yaml.safe_load(f)

    for idx, msg in enumerate(messages):
        content = msg.get("content", "").strip()
        if content:
            rows.append({
                "text": content,
                "target_language": TARGET_LANGUAGE,
                "source_file": source_rel,
                "output_file": output_rel,
                "message_index": idx,
                "role": msg["role"],
            })

translation_dataset = pd.DataFrame(rows)
print(f"Total messages to translate: {len(translation_dataset)}")
print(f"\nBreakdown by file:")
for name, group in translation_dataset.groupby("source_file"):
    print(f"  {name}: {len(group)} messages")

In [ ]:
# Run the Prompt Translation Flow on the full dataset
print(f"Translating {len(translation_dataset)} messages to {TARGET_LANGUAGE}...\n")

translated_dataset = translation_flow.generate(
    translation_dataset, max_concurrency=max_concurrency
)

print(f"\nTranslation complete: {len(translated_dataset)} messages translated")

In [ ]:
# Post-process: reconstruct translated YAML files
# Custom YAML dumper that uses block scalar style (|) for multi-line strings
class BlockStyleDumper(yaml.SafeDumper):
    pass

def _str_representer(dumper, data):
    if "\n" in data:
        return dumper.represent_scalar("tag:yaml.org,2002:str", data, style="|")
    return dumper.represent_scalar("tag:yaml.org,2002:str", data)

BlockStyleDumper.add_representer(str, _str_representer)


def _clean_content(text):
    """Strip trailing whitespace from each line.

    PyYAML cannot represent trailing spaces in block scalar (|) style and
    falls back to quoted strings. Trailing spaces are meaningless in prompt
    text, so stripping them is safe and keeps the output readable.
    """
    return "\n".join(line.rstrip() for line in text.split("\n"))


def write_translated_prompt_yaml(group_df, source_path, output_path):
    """Reconstruct a translated prompt YAML file from flow output."""
    # Load original to preserve structure for any untranslated messages
    with open(source_path) as f:
        original_messages = yaml.safe_load(f)

    translated_messages = []
    for idx, msg in enumerate(original_messages):
        translated_msg = dict(msg)
        # Find the translated content for this message index
        match = group_df[group_df["message_index"] == idx]
        if not match.empty:
            translated_msg["content"] = _clean_content(match.iloc[0]["translated_text"])
        translated_messages.append(translated_msg)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w") as f:
        yaml.dump(
            translated_messages, f,
            Dumper=BlockStyleDumper,
            default_flow_style=False,
            allow_unicode=True,
            width=120,
            sort_keys=False,
        )
    return output_path


# Write all translated prompt files
for output_rel, group in translated_dataset.groupby("output_file"):
    # Find the corresponding source file
    source_rel = group.iloc[0]["source_file"]
    source_path = SOURCE_DIR / source_rel
    output_path = OUTPUT_DIR / output_rel

    write_translated_prompt_yaml(group, source_path, output_path)
    print(f"  Saved: {output_rel}")

print(f"\nAll {len(all_prompts)} prompt files translated and saved")

## Step 2: Create Translated Flow YAML Files

Flow YAML files define the pipeline structure. We update metadata (name, ID, description, tags) and `prompt_config_path` references. The block pipeline structure remains identical.

In [ ]:
def adapt_flow_yaml(source_path, output_path):
    """Adapt a flow YAML for the target language.

    Updates metadata and prompt_config_path references.
    Block pipeline structure is preserved exactly.
    """
    with open(source_path) as f:
        flow_def = yaml.safe_load(f)

    flow_def = copy.deepcopy(flow_def)
    meta = flow_def["metadata"]

    # Update metadata
    meta["name"] = f"{meta['name']} ({TARGET_LANGUAGE})"
    meta["id"] = f"{meta['id']}-{LANGUAGE_CODE}"

    if "description" in meta:
        desc = meta["description"]
        if desc.endswith("."):
            desc = desc[:-1]
        meta["description"] = f"{desc} in {TARGET_LANGUAGE}."

    if "tags" in meta:
        lang_tag = TARGET_LANGUAGE.lower()
        if lang_tag not in meta["tags"]:
            meta["tags"].append(lang_tag)

    if "dataset_requirements" in meta and "description" in meta["dataset_requirements"]:
        req_desc = meta["dataset_requirements"]["description"]
        req_desc = req_desc.replace(
            "Input dataset should contain documents",
            f"Input dataset should contain {TARGET_LANGUAGE} documents",
        )
        meta["dataset_requirements"]["description"] = req_desc

    # Update prompt_config_path references in blocks
    for block in flow_def.get("blocks", []):
        config = block.get("block_config", {})
        if "prompt_config_path" in config:
            old_path = config["prompt_config_path"]
            new_path = old_path.replace(".yaml", f"_{LANGUAGE_CODE}.yaml")
            config["prompt_config_path"] = new_path

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w") as f:
        yaml.dump(
            flow_def, f,
            default_flow_style=False,
            allow_unicode=True,
            width=120,
            sort_keys=False,
        )

    print(f"  {output_path.relative_to(OUTPUT_DIR)}")
    print(f"    Name: {meta['name']}")
    print(f"    ID:   {meta['id']}")

In [ ]:
# Create all translated flow files
print(f"Creating {TARGET_LANGUAGE} flow definitions:\n")

for flow_rel in FLOW_FILES:
    source_path = SOURCE_DIR / flow_rel
    output_path = OUTPUT_DIR / flow_rel
    adapt_flow_yaml(source_path, output_path)
    print()

print(f"All {len(FLOW_FILES)} flow files created")

## Step 3: Verify

Check that the new flows are discoverable by the registry.

In [ ]:
# Register the output directory so FlowRegistry can discover the new flows
FlowRegistry.register_search_path(str(OUTPUT_BASE))

# Re-discover flows to pick up the new ones
FlowRegistry._entries = {}  # Reset registry cache
FlowRegistry.discover_flows()

# Search for the new language flows
lang_flows = FlowRegistry.search_flows(tag=TARGET_LANGUAGE.lower())
print(f"\nFound {len(lang_flows)} {TARGET_LANGUAGE} flows:")
for f in lang_flows:
    print(f"  - {f['name']} (ID: {f['id']})")

assert len(lang_flows) == 4, f"Expected 4 flows, found {len(lang_flows)}"
print(f"\nAll 4 {TARGET_LANGUAGE} flows are registered and discoverable")

In [ ]:
# List all generated files
print(f"Files created in {OUTPUT_DIR}:\n")
for p in sorted(OUTPUT_DIR.rglob("*.yaml")):
    print(f"  {p.relative_to(OUTPUT_DIR)}")

## Next Steps

1. **Review translated prompts** — Open the generated YAML files in the output directory and verify translation quality. Pay special attention to:
   - Domain-specific terminology
   - Template variables (`{{document}}`, etc.) are preserved
   - Parsing tags (`[QUESTION]`, `[END]`, etc.) are not translated

2. **Generate data** — Set these in your `.env` file and run `knowledge_generation.ipynb`:
   ```
   SDG_LANG=French
   TRANSLATED_FLOWS_DIR=./translated_flows
   ```

3. **Prepare seed data** — Your seed data (`SEED_DATA_PATH`) should contain documents in the target language. See the [Spanish README](../../../src/sdg_hub/flows/qa_generation/document_grounded_qa/enhanced_multi_summary_qa/multilingual/spanish/README.md) for seed data format details.

4. **Optional: contribute back** — If you'd like to contribute your translated flows to sdg_hub, copy them into `src/sdg_hub/flows/qa_generation/document_grounded_qa/enhanced_multi_summary_qa/multilingual/<language>/` and submit a PR.